# <font color=orange size=6> Packages & Libraries Init

In [14]:
import os
import numpy as np
import pandas as pd
os.environ["KERAS_BACKEND"] = "tensorflow"
import sys, sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

import keras
from keras.models import Sequential
from keras import layers
from keras.utils import pad_sequences

import tensorflow as tf

import nltk
from nltk.tokenize import RegexpTokenizer, TreebankWordTokenizer, TweetTokenizer, TreebankWordDetokenizer
from nltk.corpus import stopwords
import re

import matplotlib
from matplotlib import pyplot as plt
import matplotlib.patches as mpatches

import seaborn as sns


# 1- <font color=orange size=6> Exploratory Data Analysis

In [61]:
train_df = pd.read_csv('./train.csv')
test_df = pd.read_csv('./test.csv')
validation_df = pd.read_csv('./validation.csv')
mapping_df = pd.read_csv('./label_mapping.csv')
print(train_df.columns)
print(train_df.shape)
print(test_df.columns)
print(test_df.shape)
print(validation_df.columns)
print(validation_df.shape)
print(mapping_df.shape)

Index(['id', 'question', 'category'], dtype='str')
(58135, 3)
Index(['id', 'question'], dtype='str')
(7267, 2)
Index(['id', 'question', 'category'], dtype='str')
(7267, 3)
(10, 2)


In [62]:
print(train_df.info())
print(test_df.info())
print(validation_df.info())


<class 'pandas.DataFrame'>
RangeIndex: 58135 entries, 0 to 58134
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        58135 non-null  int64
 1   question  58135 non-null  str  
 2   category  58135 non-null  str  
dtypes: int64(1), str(2)
memory usage: 1.3 MB
None
<class 'pandas.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        7267 non-null   int64
 1   question  7267 non-null   str  
dtypes: int64(1), str(1)
memory usage: 113.7 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        7267 non-null   int64
 1   question  7267 non-null   str  
 2   category  7267 non-null   str  
dtypes: int64(1), str(2)
memory usage: 170.4 KB
None


In [63]:
train_df.dtypes

id          int64
question      str
category      str
dtype: object

In [64]:
train_df.sample()


,id,question,category
21961,21058,هل هو آمن استخدام عشبة كف مريم مع وجود اللولب,أمراض نسائية


In [65]:
cats = train_df['category'].nunique()
print(cats)

14


In [66]:
train_df['category'].isnull().sum()

np.int64(0)

In [67]:
pd.Series(train_df['category'].unique())

0       أمراض الغدد الصماء
1               الطب العام
2               أمراض الدم
3             أمراض نسائية
4     امراض الجهاز التنفسي
5          الأمراض الجلدية
6             امراض العيون
7            امراض الاطفال
8               امراض الدم
9              صحة الاسنان
10           أمراض الأطفال
11    أمراض الجهاز التنفسي
12         ارتفاع ضغط الدم
13             صحة الأسنان
dtype: str

In [68]:
train_df['category'].value_counts()

category
امراض العيون            7988
الأمراض الجلدية         7971
الطب العام              7954
أمراض نسائية            7948
أمراض الغدد الصماء      7462
امراض الاطفال           5136
ارتفاع ضغط الدم         3999
أمراض الأطفال           2850
صحة الاسنان             2355
امراض الدم              1334
امراض الجهاز التنفسي     954
صحة الأسنان              922
أمراض الجهاز التنفسي     663
أمراض الدم               599
Name: count, dtype: int64

In [69]:
train_df['category'].value_counts().sum()

np.int64(58135)

<font color="#5EEDC8" size = 4> لاحظنا أنه لا يوجد قيم فارغة في البيانات بالإضافة إلى أنه يوجد تصنيفات مكررة تختلف ببعض الأحرف
<font color="#5EEDC8"> (أمراض الأطفال - صحة الأسنان - أمراض الدم - أمراض الجهاز التنفسي)

In [70]:
train_df.head(5)

,id,question,category
0,5034,العمر ٤٢ سنة مريض ضغط باخد بيزوكارد10 اشعر بإر...,أمراض الغدد الصماء
1,44388,اعاني من فقر الدم . هل يمكنني ان القح نفسي بحق...,الطب العام
2,37246,منذ حوالي خمسة أشهر، لاحظت نزول دم مع البراز ل...,الطب العام
3,3785,انا فتاه غير متزوجة عمري 34 سنه وزني 75 سنه وط...,أمراض الدم
4,21193,تاخرت الدورة 14 يوم وعملت اختبار دم اليوم طلع ...,أمراض نسائية


In [71]:
# Compare original vs. new (rows, columns)
print("Before:", train_df.shape)
print("After: ", train_df.dropna().shape)

# Calculate exact number of rows that will be dropped
rows_dropped = len(train_df) - len(train_df.dropna())
print(f"Rows to be dropped: {rows_dropped}")

Before: (58135, 3)
After:  (58135, 3)
Rows to be dropped: 0


In [72]:
train_df[train_df['category'] == 'أمراض الجهاز التنفسي'] # تنفسي - دم - أطفال - أسنان

,id,question,category
69,108,سلام عليكم ، احتاج لايجاب جزاكم الله خيرا ، ان...,أمراض الجهاز التنفسي
229,318,منذ يومان بدات معي مشكلة صعوبة بالتنفس مع العل...,أمراض الجهاز التنفسي
293,1946,حدثت لي كحة معها دم كثير وفاتح ومعه فقاعات هذه...,أمراض الجهاز التنفسي
294,362,مرحبا.\r\nاعاني منذو فترة طويلة من ضيق التنفس ...,أمراض الجهاز التنفسي
652,1718,اعانى من ضيق فى التنفس مستمر مع الام فى الصدر ...,أمراض الجهاز التنفسي
...,...,...,...
57687,1004,اعاني من ضيق وبعدها تحول معة خفقان وفحطة راجعت...,أمراض الجهاز التنفسي
57711,823,فحصت السبت واليوم طلعت النتيجة انا سلبي وابني ...,أمراض الجهاز التنفسي
57756,154,أنا مدخن منذ ٤ سنوات يوميا علبة سجائر واحدة وم...,أمراض الجهاز التنفسي
57846,1929,مرحبا دكتور لي صديق مصاب بالتهاب رئوي مدة سنة...,أمراض الجهاز التنفسي


In [73]:
train_df[train_df['category'] == 'امراض الجهاز التنفسي']

,id,question,category
6,164,اعاني بالم ف اسفل الثدي الايسر و تكون نغزه شد...,امراض الجهاز التنفسي
25,1516,تحاليلي سليمة ولله الحمد و اشعة الصدر سليمة و ...,امراض الجهاز التنفسي
41,1304,ضيق التنفس المفاجئ مع تنميل بالوجه والشعور بال...,امراض الجهاز التنفسي
90,1996,عندي فقاعاة هواء بالرءة هل من دواء او عملية,امراض الجهاز التنفسي
173,821,ما هي العلاجات المناسبه في حال تعرضت لفايروس ك...,امراض الجهاز التنفسي
...,...,...,...
57942,1247,استخدمت علاج سولبادين بسبب ارتفاع الحراره واحس...,امراض الجهاز التنفسي
57951,1777,عندما اكون في وضعية النوم اختنق و يصعب علي الت...,امراض الجهاز التنفسي
58034,897,يوجد اسهال خفيف بس جسمي بارد جدا وكحه مستمره ...,امراض الجهاز التنفسي
58107,1767,احيان احس الهواء مايدخل الرئه وكان في شيء يحبس...,امراض الجهاز التنفسي


In [74]:
train_df[train_df['category'] == 'أمراض الدم']

,id,question,category
3,3785,انا فتاه غير متزوجة عمري 34 سنه وزني 75 سنه وط...,أمراض الدم
77,3496,لدي وجع في اسفل صدري الايسر وكبر حجم الصدر الا...,أمراض الدم
389,3302,انا فتاة ابلغ من العمر 23 سنة ولدي نقص حاد في ...,أمراض الدم
614,2173,لو سمحتم والدى عنده 90 عاما جاله تعب وضيق تنفس...,أمراض الدم
727,3341,هل جماع الزوجة من الدبر ان كانت سليمة يسبب مرض...,أمراض الدم
...,...,...,...
57691,2861,لدى امي لديها مرض السكري ولدبها ضعف في البصر و...,أمراض الدم
57741,3119,سيدة 47 سنه تعاني من نقص حاد في الحديد 9.7 \r\...,أمراض الدم
57856,3812,السلام عليكم\r\nاشكركم على جهودكم المبذوله .. ...,أمراض الدم
57983,3695,انا اعانى من شحوب فى الوجه واسوداد تحت العينين...,أمراض الدم


In [75]:
train_df[train_df['category'] == 'امراض الدم']

,id,question,category
21,3025,عملت تحليل دم ومكان الابرة لونه ازرق كتيير ومؤ...,امراض الدم
51,4153,هل استطيع الزواج بنسبه هيموقلوبين ضعيفه كماذكر...,امراض الدم
56,3020,السلام عليكم \nماهو المعدل الطبيعي لتحليل الفص...,امراض الدم
72,4162,ما هي البنودالتي؟ والتي تنص عليها نظرية الخلية...,امراض الدم
317,3889,السلام عليكم و رحمة الله و بركاتة\r\n انا ب...,امراض الدم
...,...,...,...
57975,3994,كيف نعالج مرض تلوث الدم؟,امراض الدم
58053,2535,قمت بفحص s.pyruvate وكانت نتيجة28.0 mmol/lعلما...,امراض الدم
58057,4329,ما تأثير نقص الصفائح الدموية على الحياة الزوجي...,امراض الدم
58076,2766,هل ميكروب الدم يؤدى الى الوفاة,امراض الدم


In [76]:
train_df[train_df['category'] == 'أمراض الأطفال']

,id,question,category
42,47626,دكتور الاطفال قالي اعطي ابني نص قطارة ڤيدروب ا...,أمراض الأطفال
47,51435,بنتي تتقيء في الليل اثناء النوم فقط. علما ان ه...,أمراض الأطفال
78,53059,لو سمحت طفلي حركة عينة اثناء النوم مع تشنجات و...,أمراض الأطفال
106,46462,فطريات في فم رضيع عمر ٩ أيام مع وجود نقط بيضاء...,أمراض الأطفال
116,51655,افضل حليب صناعى يمكن الاعتماد عليه لطفل عمره س...,أمراض الأطفال
...,...,...,...
58033,55479,عندي طفل عمره ثلاث سنوات كان يتحدث طبيعي وبعده...,أمراض الأطفال
58059,50895,طفلتي عمرها ثلاثة أشهر تعاني من الحازوقة ماهو ...,أمراض الأطفال
58067,51149,ابنتي عندها شهر و 10 ايام وعندها نزلة بد وسع...,أمراض الأطفال
58088,55797,ابني عمره سنة ونسبع شهور ، يخرج براز مرة واحدة...,أمراض الأطفال


In [77]:
train_df[train_df['category'] == 'امراض الاطفال']

,id,question,category
13,49260,بنتي اعطوها جرعه من لقاح كبد فايروسي اول يوم م...,امراض الاطفال
31,51291,عندي فيال مضاد حيوي(سيفترايكسون) ١جم هل يجوز ا...,امراض الاطفال
39,49821,ابني عمره ٦ شهور ونصف متشخص حساسيه البان والحا...,امراض الاطفال
52,52417,سلام ابني يبلغ من العمر ٦ سنوات عنده عاده قضم ...,امراض الاطفال
62,53453,طفلي عمره سنه ونص يستفرغ بعد الأكل وبعد البكا,امراض الاطفال
...,...,...,...
58042,53801,اختي عمرها 5سنوات شربت صابون سائل .هل شربها لل...,امراض الاطفال
58054,46482,السلام عليكم\nبنتي عندها 3شهور\nوعندها انتفاخ ...,امراض الاطفال
58078,50318,بنتي عمرها 8 شهور ببطنها طالع هالشي من وش ووش ...,امراض الاطفال
58117,49751,عندي طفل رضيع عمره شهر واسبوع تقريبا \nمعاه غا...,امراض الاطفال


In [78]:
train_df[train_df['category'] == 'صحة الأسنان']

,id,question,category
98,70668,عمري 17 ذهبت لطبيب الاسنان بسبب الم بسيط .وقال...,صحة الأسنان
387,70401,أضراس العقل العلوية مائلة للخارج وأخبرني طبيب ...,صحة الأسنان
432,69405,لدي بعض النخور في الاسنان و تم اصلاحها ولكني ا...,صحة الأسنان
450,70067,السلام عليكم\nانا قمت بخلع سني منذ حوالي ٥٥ سا...,صحة الأسنان
483,69770,مرحبا انا بعاني من حساسية الاسنان من ابتدى رمض...,صحة الأسنان
...,...,...,...
57973,70592,هل تستدعي الإجراءات الجراحية للأسنان التي خضعت...,صحة الأسنان
58027,70818,هل من الطبيعي أن اشعر برائحه وطعم في فمي لكن غ...,صحة الأسنان
58073,71491,أنا فتاة 28 سنة منذ 16سنة قمت بنزع احد الانياب...,صحة الأسنان
58090,72236,بسم الله أنا أبلغ من العمر 20 سنة أعاني من مشك...,صحة الأسنان


In [79]:
train_df[train_df['category'] == 'صحة الاسنان']

,id,question,category
22,70398,زرت طبيب الاسنان بغرض تصليح سني لكنه لم يقم يق...,صحة الاسنان
46,69526,هل تكبر الأسنان وكيف تكبر و إلى أي عمر تتوقف,صحة الاسنان
87,70323,سني امامي يولمني كتير بعد الحشوه,صحة الاسنان
109,69511,انا ركبت عدسات الايماكس قبل فتره والان اشم رائ...,صحة الاسنان
112,71303,كيف نتخلص من البلاك بين الاسنان. \r\nوايظا الث...,صحة الاسنان
...,...,...,...
57949,71435,اعاني من نزيف في اللثه,صحة الاسنان
57972,70421,بنتي عمرها 8 اشهر لا تسطيع الجلوس ولا طلع لها ...,صحة الاسنان
58017,72067,ما اسباب نزيف اللثة وما علاجها,صحة الاسنان
58039,69272,السلام عليكم اريد تركيب سن امامية بماذا تنصحني...,صحة الاسنان


<font color="#5EECD8"> imbalanced data نلاحظ وجود مشكلة 

In [80]:
category_name = "أمراض الجهاز التنفسي" # تنفسي - دم - أطفال - أسنان

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question
2065,393,أعالج عند طبيب الأمراض الصدرية عن حساسية القصب...
35808,394,أعالج عند طبيب الأمراض الصدرية عن حساسية القصب...


In [81]:
category_name = "امراض الجهاز التنفسي"

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question
9450,1735,اعاني من ضيق تنفس يرافقه الم مزمن في الظهر وال...
39118,1730,اعاني من ضيق تنفس يرافقه الم مزمن في الظهر وال...


In [82]:
category_name = "أمراض الدم"

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question


In [83]:
category_name = "امراض الدم"

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question


In [84]:
category_name = "أمراض الأطفال"

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question
12089,52196,طفلتي عمرها ١٠ شهور تعاني من قلة النوم ليلا ك...
17108,52197,طفلتي عمرها ١٠ شهور تعاني من قلة النوم ليلا ك...
4024,52427,طفلي عمره سنتين وشهرين يعاني من الاسهال اصبح ي...
18483,52426,طفلي عمره سنتين وشهرين يعاني من الاسهال اصبح ي...
39194,50198,مرحبا طفلي في 13شهر سقط الجمعة من على سرير ارت...
54017,50204,مرحبا طفلي في 13شهر سقط الجمعة من على سرير ارت...


In [85]:
category_name = "امراض الاطفال"

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question
27540,49370,ابني عمرو ٤ شهور والو يومين زي اللي فيه بلغم ب...
40167,49363,ابني عمرو ٤ شهور والو يومين زي اللي فيه بلغم ب...
41012,50791,ابني يعاني من انسداد الانف مع سيلان مع سعال قو...
55214,50790,ابني يعاني من انسداد الانف مع سيلان مع سعال قو...
14924,50749,بنتي عندها سنتين وعندها رشح علي طول وكحه مصحوب...
55962,50750,بنتي عندها سنتين وعندها رشح علي طول وكحه مصحوب...


In [86]:
category_name = "صحة الأسنان"

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question


In [87]:
category_name = "صحة الاسنان"

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question


<font color="#5EEDC8" size = 4> نلاحظ وجود تكرارات في الأسئلة في كل من (أمراض الجهاز التنفسي - أمراض الأطفال)

<font color="#5EEDC8" size=4>نقوم توحيد التصنيفات المتشابهة (التي تختلف بالهمزة)

In [109]:
train_df["category"] = train_df["category"].str.strip().replace({
    "امراض الجهاز التنفسي": "أمراض الجهاز التنفسي"
})
train_df["category"] = train_df["category"].str.strip().replace({
    "امراض الدم": "أمراض الدم"
})
train_df["category"] = train_df["category"].str.strip().replace({
    "امراض الاطفال": "أمراض الأطفال"
})
train_df["category"] = train_df["category"].str.strip().replace({
    "صحة الاسنان": "صحة الأسنان"
})
train_df["category"] = train_df["category"].str.strip().replace({
    "امراض العيون": "أمراض العيون"
})


In [110]:
pd.Series(train_df['category'].unique())

0      أمراض الغدد الصماء
1              الطب العام
2              أمراض الدم
3            أمراض نسائية
4    أمراض الجهاز التنفسي
5         الأمراض الجلدية
6            أمراض العيون
7           أمراض الأطفال
8             صحة الأسنان
9         ارتفاع ضغط الدم
dtype: str

<font color="#5EEDC8" size=4> هنا عدنا للبحث عن الأسئلة المتطابقة بعد دمج التصنيفات من أجل إحصاء في حال وجود تطابقات أكثر

In [ ]:
category_name = "أمراض الجهاز التنفسي"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question
2065,393,أعالج عند طبيب الأمراض الصدرية عن حساسية القصب...
35808,394,أعالج عند طبيب الأمراض الصدرية عن حساسية القصب...
9450,1735,اعاني من ضيق تنفس يرافقه الم مزمن في الظهر وال...
39118,1730,اعاني من ضيق تنفس يرافقه الم مزمن في الظهر وال...


In [ ]:
category_name = "أمراض الدم"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question


In [ ]:
category_name = "أمراض الأطفال"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question
27540,49370,ابني عمرو ٤ شهور والو يومين زي اللي فيه بلغم ب...
40167,49363,ابني عمرو ٤ شهور والو يومين زي اللي فيه بلغم ب...
41012,50791,ابني يعاني من انسداد الانف مع سيلان مع سعال قو...
55214,50790,ابني يعاني من انسداد الانف مع سيلان مع سعال قو...
14924,50749,بنتي عندها سنتين وعندها رشح علي طول وكحه مصحوب...
55962,50750,بنتي عندها سنتين وعندها رشح علي طول وكحه مصحوب...
12089,52196,طفلتي عمرها ١٠ شهور تعاني من قلة النوم ليلا ك...
17108,52197,طفلتي عمرها ١٠ شهور تعاني من قلة النوم ليلا ك...
4024,52427,طفلي عمره سنتين وشهرين يعاني من الاسهال اصبح ي...
18483,52426,طفلي عمره سنتين وشهرين يعاني من الاسهال اصبح ي...


In [ ]:
category_name = "صحة الأسنان" #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question


In [ ]:
category_name = "أمراض العيون" #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question


In [ ]:
category_name = "الأمراض الجلدية"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question


In [ ]:
category_name = "الطب العام"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question


In [ ]:
category_name = "أمراض نسائية"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question


In [ ]:
category_name = "أمراض الغدد الصماء"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question


In [ ]:
category_name = "ارتفاع ضغط الدم"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question


<font color="#5EEDC8" size=4> نقوم حاليا بالبحث عن الأسئلة المتشابهة في كل تصنيف

In [ ]:
category_name = "أمراض الجهاز التنفسي"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط
threshold = 0.80
min_shared_words = 4

def question_words(text):
    text = str(text).strip().lower()

    # Replace common punctuation with spaces
    for char in ".,،؛:!?؟()[]{}\"'ـ":
        text = text.replace(char, " ")

    return text.split()


def longest_common_subsequence_length(words1, words2):
    previous = [0] * (len(words2) + 1)

    for word1 in words1:
        current = [0]
        for j, word2 in enumerate(words2, start=1):
            if word1 == word2:
                current.append(previous[j - 1] + 1)
            else:
                current.append(max(previous[j], current[-1]))
        previous = current

    return previous[-1]


subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).reset_index(drop=True)
ids = subset["id"].tolist()
raw_questions = subset["question"].tolist()
words = [question_words(q) for q in subset["question"]]
word_sets = [set(w) for w in words]
lengths = [len(w) for w in words]

matches = []
n = len(ids)

for i in range(n):
    len_i = lengths[i]
    if len_i < min_shared_words:
        continue
    words_i = words[i]
    set_i = word_sets[i]

    for j in range(i + 1, n):
        len_j = lengths[j]
        if len_j < min_shared_words:
            continue

        shorter_length = len_i if len_i < len_j else len_j
        required_shared = max(min_shared_words, int(threshold * shorter_length))

        # 2. Fast upper-bound filter: unordered set intersection must meet required_shared
        if len(set_i & word_sets[j]) < required_shared:
            continue

        # 3. Only run expensive LCS on the tiny fraction of pairs that pass the filter
        shared_in_order = longest_common_subsequence_length(words_i, words[j])
        shorter_coverage = shared_in_order / shorter_length

        if shared_in_order >= min_shared_words and shorter_coverage >= threshold:
            matches.append({
                "id_1": ids[i],
                "question_1": raw_questions[i],
                "id_2": ids[j],
                "question_2": raw_questions[j],
                "shared_words_in_order": shared_in_order,
                "shorter_question_coverage": round(shorter_coverage, 3),
            })

similar_questions = pd.DataFrame(matches).sort_values(
    "shorter_question_coverage",
    ascending=False
)

display(similar_questions)

,id_1,question_1,id_2,question_2,shared_words_in_order,shorter_question_coverage
0,362,مرحبا.\r\nاعاني منذو فترة طويلة من ضيق التنفس ...,1263,اعاني من ضيق في النفس,5,1.0
2,1573,انقطاع النفس اثناء النوم,1540,مرحبا، أنا سيدة عمري 34 أعاني انقطاع النفس أثن...,4,1.0
3,1573,انقطاع النفس اثناء النوم,1510,السلام عليكم ورحمة الله، هل يعتبر مرض انقطاع ا...,4,1.0
6,393,أعالج عند طبيب الأمراض الصدرية عن حساسية القصب...,394,أعالج عند طبيب الأمراض الصدرية عن حساسية القصب...,32,1.0
20,1735,اعاني من ضيق تنفس يرافقه الم مزمن في الظهر وال...,1730,اعاني من ضيق تنفس يرافقه الم مزمن في الظهر وال...,29,1.0
...,...,...,...,...,...,...
84,1263,اعاني من ضيق في النفس,1879,اعاني من ضيق تنفس منذ ٣ سنوات وذهبت لاكثر من ط...,4,0.8
83,1263,اعاني من ضيق في النفس,1857,أعاني من وجع في الصدر مع ضيق في النفس وأحيانًا...,4,0.8
82,1263,اعاني من ضيق في النفس,1098,اعاني من محرارة و ضيق في التنفس مع عدم النوم و...,4,0.8
88,1801,اعاني من اختناق تنفس في الليل اثناء النوم\r\nم...,1518,السلام عليكم، اعاني من ضعف تنفس أي أشعر بصعوبة...,8,0.8


In [ ]:
category_name = "أمراض الدم"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط
threshold = 0.80
min_shared_words = 4

def question_words(text):
    text = str(text).strip().lower()

    # Replace common punctuation with spaces
    for char in ".,،؛:!?؟()[]{}\"'ـ":
        text = text.replace(char, " ")

    return text.split()


def longest_common_subsequence_length(words1, words2):
    previous = [0] * (len(words2) + 1)

    for word1 in words1:
        current = [0]
        for j, word2 in enumerate(words2, start=1):
            if word1 == word2:
                current.append(previous[j - 1] + 1)
            else:
                current.append(max(previous[j], current[-1]))
        previous = current

    return previous[-1]


subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).reset_index(drop=True)
ids = subset["id"].tolist()
raw_questions = subset["question"].tolist()
words = [question_words(q) for q in subset["question"]]
word_sets = [set(w) for w in words]
lengths = [len(w) for w in words]

matches = []
n = len(ids)

for i in range(n):
    len_i = lengths[i]
    if len_i < min_shared_words:
        continue
    words_i = words[i]
    set_i = word_sets[i]

    for j in range(i + 1, n):
        len_j = lengths[j]
        if len_j < min_shared_words:
            continue

        shorter_length = len_i if len_i < len_j else len_j
        required_shared = max(min_shared_words, int(threshold * shorter_length))

        # 2. Fast upper-bound filter: unordered set intersection must meet required_shared
        if len(set_i & word_sets[j]) < required_shared:
            continue

        # 3. Only run expensive LCS on the tiny fraction of pairs that pass the filter
        shared_in_order = longest_common_subsequence_length(words_i, words[j])
        shorter_coverage = shared_in_order / shorter_length

        if shared_in_order >= min_shared_words and shorter_coverage >= threshold:
            matches.append({
                "id_1": ids[i],
                "question_1": raw_questions[i],
                "id_2": ids[j],
                "question_2": raw_questions[j],
                "shared_words_in_order": shared_in_order,
                "shorter_question_coverage": round(shorter_coverage, 3),
            })

similar_questions = pd.DataFrame(matches).sort_values(
    "shorter_question_coverage",
    ascending=False
)

display(similar_questions)

,id_1,question_1,id_2,question_2,shared_words_in_order,shorter_question_coverage
15,3340,ماهو علاج فقر الدم الطبيعي من دون ادويه,3111,ماهو علاج فقر الدم,4,1.000
8,3765,ما هو مرض الثلاسيميا,3390,ما هو علاج مرض الثلاسيميا,4,1.000
37,3111,ماهو علاج فقر الدم,3113,ماهو علاج فقر الدم للحامل,4,1.000
44,2525,زياده عدد كريات الدم البيضاء,4395,عندي زياده كبيره في عدد الكريات البيضاء مع تضخ...,5,1.000
26,4426,ما هي مجموعة عوامل التجلط في الإنسان؟.؟\r\n\r\...,4436,ما هي مجموعة عوامل التجلط في الإنسان؟.,7,1.000
28,3074,مرحبا \r\nماهي اعراض فقر الدم؟؟,3962,ماهي اعراض فقر الدم؟,4,1.000
30,3648,ماذا يعني ان يصل مستوى كريات الدم البيضاء الى ...,3647,ماذا يعني ان يصل مستوى كريات الدم البيضاء في ا...,18,1.000
19,3068,السلام عليكم \nماهو علاج فقر الدم لطفله عمرها ...,3111,ماهو علاج فقر الدم,4,1.000
35,3372,ما أعراض فقر الدم؟,2902,ما هي أعراض فقر الدم؟ مادا يسبب ان لم يعالج؟,4,1.000
36,3372,ما أعراض فقر الدم؟,3371,ما أعراض فقر الدم بشكل عام؟ وما أعراض نقص الحد...,4,1.000


In [ ]:
category_name = "أمراض الأطفال"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط
threshold = 0.80
min_shared_words = 4

def question_words(text):
    text = str(text).strip().lower()

    # Replace common punctuation with spaces
    for char in ".,،؛:!?؟()[]{}\"'ـ":
        text = text.replace(char, " ")

    return text.split()


def longest_common_subsequence_length(words1, words2):
    previous = [0] * (len(words2) + 1)

    for word1 in words1:
        current = [0]
        for j, word2 in enumerate(words2, start=1):
            if word1 == word2:
                current.append(previous[j - 1] + 1)
            else:
                current.append(max(previous[j], current[-1]))
        previous = current

    return previous[-1]


subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).reset_index(drop=True)


ids = subset["id"].tolist()
raw_questions = subset["question"].tolist()
words = [question_words(q) for q in subset["question"]]
word_sets = [set(w) for w in words]
lengths = [len(w) for w in words]

matches = []
n = len(ids)

for i in range(n):
    len_i = lengths[i]
    if len_i < min_shared_words:
        continue
    words_i = words[i]
    set_i = word_sets[i]

    for j in range(i + 1, n):
        len_j = lengths[j]
        if len_j < min_shared_words:
            continue

        shorter_length = len_i if len_i < len_j else len_j
        required_shared = max(min_shared_words, int(threshold * shorter_length))

        # 2. Fast upper-bound filter: unordered set intersection must meet required_shared
        if len(set_i & word_sets[j]) < required_shared:
            continue

        # 3. Only run expensive LCS on the tiny fraction of pairs that pass the filter
        shared_in_order = longest_common_subsequence_length(words_i, words[j])
        shorter_coverage = shared_in_order / shorter_length

        if shared_in_order >= min_shared_words and shorter_coverage >= threshold:
            matches.append({
                "id_1": ids[i],
                "question_1": raw_questions[i],
                "id_2": ids[j],
                "question_2": raw_questions[j],
                "shared_words_in_order": shared_in_order,
                "shorter_question_coverage": round(shorter_coverage, 3),
            })

similar_questions = pd.DataFrame(matches).sort_values(
    "shorter_question_coverage",
    ascending=False
)

display(similar_questions)

,id_1,question_1,id_2,question_2,shared_words_in_order,shorter_question_coverage
2,48459,طفلة عمرها 3 سنوات ونصف نتيجة فحص الدم cbc كان...,48461,طفلة عمرها 3 سنوات ونصف نتيجة فحص الدم cbc كان...,47,1.000
5,50492,بنتي عندها ٣شهور و ٢٣يوم وزنها ٥كيلو لكن الطول...,50490,بنتي عندها ٣شهور و ٢٣يوم وزنها ٥كيلو حاليا ،لك...,21,1.000
19,52196,طفلتي عمرها ١٠ شهور تعاني من قلة النوم ليلا ك...,52197,طفلتي عمرها ١٠ شهور تعاني من قلة النوم ليلا ك...,33,1.000
23,50749,بنتي عندها سنتين وعندها رشح علي طول وكحه مصحوب...,50750,بنتي عندها سنتين وعندها رشح علي طول وكحه مصحوب...,15,1.000
21,48471,السلام عليكم ورحمة الله وبركاته..\n\nطفلي يتم ...,48470,السلام عليكم ورحمة الله وبركاته..\n\nطفلي يتم ...,30,1.000
17,56102,افضل حليب صناعي للرضع؟,46874,طفلي لا يرضع طبيعي يرضع صناعي فقط واستخدم حليب...,4,1.000
9,52427,طفلي عمره سنتين وشهرين يعاني من الاسهال اصبح ي...,52426,طفلي عمره سنتين وشهرين يعاني من الاسهال اصبح ي...,34,1.000
11,50936,علما ان الطفل يرضع حليب طبي فقط لانه لديه حساس...,51029,لديه حساسيه من الحليب,4,1.000
25,46781,من فضلكم طمنوني اذا توجد مضاعفات واذا الامر خط...,46936,من فضلكم طمنوني اذا توجد مضاعفات واذا الامر خط...,29,1.000
52,50791,ابني يعاني من انسداد الانف مع سيلان مع سعال قو...,50790,ابني يعاني من انسداد الانف مع سيلان مع سعال قو...,16,1.000


In [ ]:
category_name = "صحة الأسنان"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط
threshold = 0.80
min_shared_words = 4

def question_words(text):
    text = str(text).strip().lower()

    # Replace common punctuation with spaces
    for char in ".,،؛:!?؟()[]{}\"'ـ":
        text = text.replace(char, " ")

    return text.split()


def longest_common_subsequence_length(words1, words2):
    previous = [0] * (len(words2) + 1)

    for word1 in words1:
        current = [0]
        for j, word2 in enumerate(words2, start=1):
            if word1 == word2:
                current.append(previous[j - 1] + 1)
            else:
                current.append(max(previous[j], current[-1]))
        previous = current

    return previous[-1]


subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).reset_index(drop=True)
ids = subset["id"].tolist()
raw_questions = subset["question"].tolist()
words = [question_words(q) for q in subset["question"]]
word_sets = [set(w) for w in words]
lengths = [len(w) for w in words]

matches = []
n = len(ids)

for i in range(n):
    len_i = lengths[i]
    if len_i < min_shared_words:
        continue
    words_i = words[i]
    set_i = word_sets[i]

    for j in range(i + 1, n):
        len_j = lengths[j]
        if len_j < min_shared_words:
            continue

        shorter_length = len_i if len_i < len_j else len_j
        required_shared = max(min_shared_words, int(threshold * shorter_length))

        # 2. Fast upper-bound filter: unordered set intersection must meet required_shared
        if len(set_i & word_sets[j]) < required_shared:
            continue

        # 3. Only run expensive LCS on the tiny fraction of pairs that pass the filter
        shared_in_order = longest_common_subsequence_length(words_i, words[j])
        shorter_coverage = shared_in_order / shorter_length

        if shared_in_order >= min_shared_words and shorter_coverage >= threshold:
            matches.append({
                "id_1": ids[i],
                "question_1": raw_questions[i],
                "id_2": ids[j],
                "question_2": raw_questions[j],
                "shared_words_in_order": shared_in_order,
                "shorter_question_coverage": round(shorter_coverage, 3),
            })

similar_questions = pd.DataFrame(matches).sort_values(
    "shorter_question_coverage",
    ascending=False
)

display(similar_questions)

,id_1,question_1,id_2,question_2,shared_words_in_order,shorter_question_coverage
3,70622,اسناني تتاكل واغلبها بدون عصب كيف احافظ على ال...,72162,كيف احافظ على الاسنان,4,1.0
2,70374,اعاني من نزيف في اللثه عند استخدام خيط الاسنان...,71435,اعاني من نزيف في اللثه,5,1.0
13,71635,ماهو أفضل معجون أسنان؟,69769,ماهو أفضل معجون أسنان بالصيدليات والذي يحافظ ع...,4,1.0
17,70707,الم في الاسنان ما الحل,70738,الم في المفصل الايمن للفك يزيد عند النوم على ج...,5,1.0
12,71635,ماهو أفضل معجون أسنان؟,71215,ماهو أفضل معجون أسنان يوصي به ،، للتبييض وإعطا...,4,1.0
...,...,...,...,...,...,...
78,70008,السلام عليكم ،،،، اعاني من انتفاخ في اللثه من ...,71435,اعاني من نزيف في اللثه,4,0.8
81,71084,ما سبب الكلس في الأسنان؟,71998,ما سبب الشعور بألم شديد في الأسنان عند شرب الم...,4,0.8
80,69014,السلام عليكم اعاني من نزيف في اللثة كل صباح لا...,71435,اعاني من نزيف في اللثه,4,0.8
79,69777,مرحبا اعاني من سن لم ينمو ومن تورم في اللثه ال...,71435,اعاني من نزيف في اللثه,4,0.8


In [117]:
category_name = "أمراض العيون"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط
threshold = 0.80
min_shared_words = 4

def question_words(text):
    text = str(text).strip().lower()

    # Replace common punctuation with spaces
    for char in ".,،؛:!?؟()[]{}\"'ـ":
        text = text.replace(char, " ")

    return text.split()


def longest_common_subsequence_length(words1, words2):
    previous = [0] * (len(words2) + 1)

    for word1 in words1:
        current = [0]
        for j, word2 in enumerate(words2, start=1):
            if word1 == word2:
                current.append(previous[j - 1] + 1)
            else:
                current.append(max(previous[j], current[-1]))
        previous = current

    return previous[-1]


subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).reset_index(drop=True)
ids = subset["id"].tolist()
raw_questions = subset["question"].tolist()
words = [question_words(q) for q in subset["question"]]
word_sets = [set(w) for w in words]
lengths = [len(w) for w in words]

matches = []
n = len(ids)

for i in range(n):
    len_i = lengths[i]
    if len_i < min_shared_words:
        continue
    words_i = words[i]
    set_i = word_sets[i]

    for j in range(i + 1, n):
        len_j = lengths[j]
        if len_j < min_shared_words:
            continue

        shorter_length = len_i if len_i < len_j else len_j
        required_shared = max(min_shared_words, int(threshold * shorter_length))

        # 2. Fast upper-bound filter: unordered set intersection must meet required_shared
        if len(set_i & word_sets[j]) < required_shared:
            continue

        # 3. Only run expensive LCS on the tiny fraction of pairs that pass the filter
        shared_in_order = longest_common_subsequence_length(words_i, words[j])
        shorter_coverage = shared_in_order / shorter_length

        if shared_in_order >= min_shared_words and shorter_coverage >= threshold:
            matches.append({
                "id_1": ids[i],
                "question_1": raw_questions[i],
                "id_2": ids[j],
                "question_2": raw_questions[j],
                "shared_words_in_order": shared_in_order,
                "shorter_question_coverage": round(shorter_coverage, 3),
            })

similar_questions = pd.DataFrame(matches).sort_values(
    "shorter_question_coverage",
    ascending=False
)

display(similar_questions)

,id_1,question_1,id_2,question_2,shared_words_in_order,shorter_question_coverage
456,63140,ماهو سبب نقص حجم العين بعد اجراء عملية نزع الم...,63126,عملية نزع المياه من العين,5,1.0
416,64125,اريد قطرة منظفة للعين ولبياض العين بسبب الارها...,64106,اريد قطرة منظفة للعين ومرطبة ولبياض العين بسبب...,21,1.0
124,63916,اعاني من القرنية المخروطية واستخدم العدسات ال...,63684,اعاني من احمرار في العين,5,1.0
17,58482,السلام عليكم ورحمه الله\r\nاعاني من احمرار الع...,63684,اعاني من احمرار في العين,5,1.0
198,56910,السلام عليكم عندي كشف مقاس النظاره وابغى اخذ ع...,58392,السلام عليكم \r\nعندي كشف مقاس النظاره وابغى ا...,14,1.0
...,...,...,...,...,...,...
20,62468,توسع حدقي كامل في العين اليسرى بشكل مفاجئ مترا...,62469,حدث معي البارحة توسع كامل مفاجئ في حدقة العين ...,20,0.8
34,61002,اعاني من الم في العين اليسرى وقمت بفحصها واخبر...,64779,اعانى. من الم. في العين؟,4,0.8
22,60702,منذ اكثر من شهر و نصف اعاني من جفاف في العين و...,63684,اعاني من احمرار في العين,4,0.8
23,60404,اعاني من الحرقة في العين و الاحمرار و البزة في...,63684,اعاني من احمرار في العين,4,0.8


In [118]:
category_name = "الأمراض الجلدية"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط
threshold = 0.80
min_shared_words = 4

def question_words(text):
    text = str(text).strip().lower()

    # Replace common punctuation with spaces
    for char in ".,،؛:!?؟()[]{}\"'ـ":
        text = text.replace(char, " ")

    return text.split()


def longest_common_subsequence_length(words1, words2):
    previous = [0] * (len(words2) + 1)

    for word1 in words1:
        current = [0]
        for j, word2 in enumerate(words2, start=1):
            if word1 == word2:
                current.append(previous[j - 1] + 1)
            else:
                current.append(max(previous[j], current[-1]))
        previous = current

    return previous[-1]


subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).reset_index(drop=True)
ids = subset["id"].tolist()
raw_questions = subset["question"].tolist()
words = [question_words(q) for q in subset["question"]]
word_sets = [set(w) for w in words]
lengths = [len(w) for w in words]

matches = []
n = len(ids)

for i in range(n):
    len_i = lengths[i]
    if len_i < min_shared_words:
        continue
    words_i = words[i]
    set_i = word_sets[i]

    for j in range(i + 1, n):
        len_j = lengths[j]
        if len_j < min_shared_words:
            continue

        shorter_length = len_i if len_i < len_j else len_j
        required_shared = max(min_shared_words, int(threshold * shorter_length))

        # 2. Fast upper-bound filter: unordered set intersection must meet required_shared
        if len(set_i & word_sets[j]) < required_shared:
            continue

        # 3. Only run expensive LCS on the tiny fraction of pairs that pass the filter
        shared_in_order = longest_common_subsequence_length(words_i, words[j])
        shorter_coverage = shared_in_order / shorter_length

        if shared_in_order >= min_shared_words and shorter_coverage >= threshold:
            matches.append({
                "id_1": ids[i],
                "question_1": raw_questions[i],
                "id_2": ids[j],
                "question_2": raw_questions[j],
                "shared_words_in_order": shared_in_order,
                "shorter_question_coverage": round(shorter_coverage, 3),
            })

similar_questions = pd.DataFrame(matches).sort_values(
    "shorter_question_coverage",
    ascending=False
)

display(similar_questions)

,id_1,question_1,id_2,question_2,shared_words_in_order,shorter_question_coverage
11,36383,هل من الممكن أن يبدأ الصلع الوراثي من عمر ١٦ ...,31703,هل من الممكن أن يبدأ الصلع الوراثي من عمر ١٦ م...,37,1.0
7,34793,اعاني من اثار حب الشباب و التي لم تزل و اريد ن...,33829,اعاني من اثار حب الشباب,5,1.0
34,30081,ما هو الجزء الأكثر أهمية في علاج التهاب الجلد ...,30084,ما هو علاج التهاب الجلد؟,5,1.0
22,36153,اعاني من تصبغات على مستوى الوجنتين مع اثار حب ...,33829,اعاني من اثار حب الشباب,5,1.0
110,31664,تساقط شعر بشكل كبير,33973,عمري 18 سنة وعندي تساقط في شعر بشكل كبير ايش ...,4,1.0
...,...,...,...,...,...,...
102,30080,ما هي أعراض الحساسية في الجلد؟,30083,ما هي أعراض الجلد للجفاف؟,4,0.8
101,26892,انتشار هذا النوع من الحبوب في الجسم تظهر كاحمر...,34247,اغاني من الحبوب في الجسم,4,0.8
100,33661,اعاني الام حادة اسفل القدم ما هو السبب علما ان...,30084,ما هو علاج التهاب الجلد؟,4,0.8
108,33851,ضهرت بثور حمراء على بطني و ضهري اثناء تناولي ا...,33850,ضهرت بثور حمراء على جلدي في بطني و في ظهري اثن...,8,0.8


In [119]:
category_name = "الطب العام"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط
threshold = 0.80
min_shared_words = 4

def question_words(text):
    text = str(text).strip().lower()

    # Replace common punctuation with spaces
    for char in ".,،؛:!?؟()[]{}\"'ـ":
        text = text.replace(char, " ")

    return text.split()


def longest_common_subsequence_length(words1, words2):
    previous = [0] * (len(words2) + 1)

    for word1 in words1:
        current = [0]
        for j, word2 in enumerate(words2, start=1):
            if word1 == word2:
                current.append(previous[j - 1] + 1)
            else:
                current.append(max(previous[j], current[-1]))
        previous = current

    return previous[-1]


subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).reset_index(drop=True)
ids = subset["id"].tolist()
raw_questions = subset["question"].tolist()
words = [question_words(q) for q in subset["question"]]
word_sets = [set(w) for w in words]
lengths = [len(w) for w in words]

matches = []
n = len(ids)

for i in range(n):
    len_i = lengths[i]
    if len_i < min_shared_words:
        continue
    words_i = words[i]
    set_i = word_sets[i]

    for j in range(i + 1, n):
        len_j = lengths[j]
        if len_j < min_shared_words:
            continue

        shorter_length = len_i if len_i < len_j else len_j
        required_shared = max(min_shared_words, int(threshold * shorter_length))

        # 2. Fast upper-bound filter: unordered set intersection must meet required_shared
        if len(set_i & word_sets[j]) < required_shared:
            continue

        # 3. Only run expensive LCS on the tiny fraction of pairs that pass the filter
        shared_in_order = longest_common_subsequence_length(words_i, words[j])
        shorter_coverage = shared_in_order / shorter_length

        if shared_in_order >= min_shared_words and shorter_coverage >= threshold:
            matches.append({
                "id_1": ids[i],
                "question_1": raw_questions[i],
                "id_2": ids[j],
                "question_2": raw_questions[j],
                "shared_words_in_order": shared_in_order,
                "shorter_question_coverage": round(shorter_coverage, 3),
            })

similar_questions = pd.DataFrame(matches).sort_values(
    "shorter_question_coverage",
    ascending=False
)

display(similar_questions)

,id_1,question_1,id_2,question_2,shared_words_in_order,shorter_question_coverage
2,37159,أجريت تحاليل شاملة وكانت جميع النتائج طبيعية ب...,44200,لا أعاني من أي أعراض,5,1.0
6,39252,لا أعاني من أي أعراض ولكن ما هو سبب اختلاف قرا...,44200,لا أعاني من أي أعراض,5,1.0
70,36877,أعاني من تساقط شعر شحوب البشرة زيادة ونقصان ف ...,44271,اريد معرفة نتيجة التحاليل,4,1.0
39,44645,ارجو الرد باسرع وقت,39837,جدتي عمرها 120عام واليوم تبين لنا انها تعاني م...,4,1.0
30,37597,لدي تعب وخمول وضعف في الذاكرة هل نتائج هذا الت...,44095,هل نتائج هذا التحليل طبيعية؟,5,1.0
...,...,...,...,...,...,...
127,39321,السلام عليكم\r\nعملت هذه التحاليل ابغى اعرف و...,40967,السلام عليكم \r\nابغى اعرف النتيجة,4,0.8
129,40512,هل نسبة السكر في الدم 110 في نهار رمضان بعد ت...,44200,لا أعاني من أي أعراض,4,0.8
128,39167,انا عمري 18 سنة في اخر سنة من الثانوية ، اعاني...,44200,لا أعاني من أي أعراض,4,0.8
132,44200,لا أعاني من أي أعراض,42689,مرحبا دكتور، أعاني من نقص في أوميغا 3 ولم أجرِ...,4,0.8


In [120]:
category_name = "أمراض نسائية"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط
threshold = 0.80
min_shared_words = 4

def question_words(text):
    text = str(text).strip().lower()

    # Replace common punctuation with spaces
    for char in ".,،؛:!?؟()[]{}\"'ـ":
        text = text.replace(char, " ")

    return text.split()


def longest_common_subsequence_length(words1, words2):
    previous = [0] * (len(words2) + 1)

    for word1 in words1:
        current = [0]
        for j, word2 in enumerate(words2, start=1):
            if word1 == word2:
                current.append(previous[j - 1] + 1)
            else:
                current.append(max(previous[j], current[-1]))
        previous = current

    return previous[-1]


subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).reset_index(drop=True)
ids = subset["id"].tolist()
raw_questions = subset["question"].tolist()
words = [question_words(q) for q in subset["question"]]
word_sets = [set(w) for w in words]
lengths = [len(w) for w in words]

matches = []
n = len(ids)

for i in range(n):
    len_i = lengths[i]
    if len_i < min_shared_words:
        continue
    words_i = words[i]
    set_i = word_sets[i]

    for j in range(i + 1, n):
        len_j = lengths[j]
        if len_j < min_shared_words:
            continue

        shorter_length = len_i if len_i < len_j else len_j
        required_shared = max(min_shared_words, int(threshold * shorter_length))

        # 2. Fast upper-bound filter: unordered set intersection must meet required_shared
        if len(set_i & word_sets[j]) < required_shared:
            continue

        # 3. Only run expensive LCS on the tiny fraction of pairs that pass the filter
        shared_in_order = longest_common_subsequence_length(words_i, words[j])
        shorter_coverage = shared_in_order / shorter_length

        if shared_in_order >= min_shared_words and shorter_coverage >= threshold:
            matches.append({
                "id_1": ids[i],
                "question_1": raw_questions[i],
                "id_2": ids[j],
                "question_2": raw_questions[j],
                "shared_words_in_order": shared_in_order,
                "shorter_question_coverage": round(shorter_coverage, 3),
            })

similar_questions = pd.DataFrame(matches).sort_values(
    "shorter_question_coverage",
    ascending=False
)

display(similar_questions)

,id_1,question_1,id_2,question_2,shared_words_in_order,shorter_question_coverage
49,14595,بدأت لدي الدورة لمدة 24 ساعة ثم انقطعت وكان ال...,14584,بدأت لدي الدورة لمدة 24 ساعة ثم انقطعت وكان ال...,26,0.963
21,68432,اذا تم الجماع يوم ١٢ ويوم التبويض كان يوم ١٨ ه...,68410,اذا تم الجماع يوم ١٢ وطول الدوره ٣٠ ويوم التبو...,21,0.955
55,15894,سويت تحليل حمل رقمي قبل الدورة بيومين في اليوم...,16865,تحليل حمل قبل الدورة بيومين هل يوجد حمل؟.,7,0.875
7,13992,قمت بعمل تحليل حمل رقمي قبل معاد الدورة ب ٣ اي...,16865,تحليل حمل قبل الدورة بيومين هل يوجد حمل؟.,7,0.875
12,68095,الم اسفل البطن يسار والم عند المشي,20290,السلام عليكم ورحمة الله وبركاته \r\nعندي الم ا...,6,0.857
18,21440,تأخرت الدورة الشهرية لشهور بسبب قلة الأكل ونزو...,21445,تأخرت الدورة الشهرية لشهور بسبب قلة الأكل ونزو...,17,0.850
5,67741,عندي سؤال انا عملت فحص لهرمون الحمل قبل ما الد...,67738,عندي سؤال انا عملت فحص لهرمون الحمل قبل ما الد...,28,0.824
3,19722,كنت اعاني من الام الظهر قبل موعد الدورة الشهري...,67634,اعاني من تأخير الدورة الشهرية,4,0.800
6,16869,هل من هذه الصورة استطيع معرفه نوع الجنين حتي و...,20578,تحديد نوع الجنين الشهر الرابع,4,0.800
4,13921,عمري 18 سنه لست متزوجة اعاني من مشاكل في الدور...,67634,اعاني من تأخير الدورة الشهرية,4,0.800


In [121]:
category_name = "أمراض الغدد الصماء"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط
threshold = 0.80
min_shared_words = 4

def question_words(text):
    text = str(text).strip().lower()

    # Replace common punctuation with spaces
    for char in ".,،؛:!?؟()[]{}\"'ـ":
        text = text.replace(char, " ")

    return text.split()


def longest_common_subsequence_length(words1, words2):
    previous = [0] * (len(words2) + 1)

    for word1 in words1:
        current = [0]
        for j, word2 in enumerate(words2, start=1):
            if word1 == word2:
                current.append(previous[j - 1] + 1)
            else:
                current.append(max(previous[j], current[-1]))
        previous = current

    return previous[-1]


subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).reset_index(drop=True)
ids = subset["id"].tolist()
raw_questions = subset["question"].tolist()
words = [question_words(q) for q in subset["question"]]
word_sets = [set(w) for w in words]
lengths = [len(w) for w in words]

matches = []
n = len(ids)

for i in range(n):
    len_i = lengths[i]
    if len_i < min_shared_words:
        continue
    words_i = words[i]
    set_i = word_sets[i]

    for j in range(i + 1, n):
        len_j = lengths[j]
        if len_j < min_shared_words:
            continue

        shorter_length = len_i if len_i < len_j else len_j
        required_shared = max(min_shared_words, int(threshold * shorter_length))

        # 2. Fast upper-bound filter: unordered set intersection must meet required_shared
        if len(set_i & word_sets[j]) < required_shared:
            continue

        # 3. Only run expensive LCS on the tiny fraction of pairs that pass the filter
        shared_in_order = longest_common_subsequence_length(words_i, words[j])
        shorter_coverage = shared_in_order / shorter_length

        if shared_in_order >= min_shared_words and shorter_coverage >= threshold:
            matches.append({
                "id_1": ids[i],
                "question_1": raw_questions[i],
                "id_2": ids[j],
                "question_2": raw_questions[j],
                "shared_words_in_order": shared_in_order,
                "shorter_question_coverage": round(shorter_coverage, 3),
            })

similar_questions = pd.DataFrame(matches).sort_values(
    "shorter_question_coverage",
    ascending=False
)

display(similar_questions)

,id_1,question_1,id_2,question_2,shared_words_in_order,shorter_question_coverage
23,6226,السلام عليكم دكتور اني عندي تساقط الشعر والم م...,10315,اعراض خمول الغدة الدرقية,4,1.0
6,6121,قبل ٣ أسابيع لاحظت بداية تورم ف جانب الرقبة ال...,7950,عندي خمول في الغدة الدرقية,5,1.0
61,10809,ما هي اعراض خمول الغدة الدرقية ؟,10315,اعراض خمول الغدة الدرقية,4,1.0
385,11192,ماهو علاج خمول الغده الدرقيه؟,9986,ماهو علاج خمول الغده,4,1.0
406,13381,ما هي أعراض الغدة,5720,كانت أمي تعاني من مشاكل في الغدة الدرقية لمدة ...,4,1.0
...,...,...,...,...,...,...
487,10836,ما هي اعراض اضطرب الغدة الدرقيه وكيف اعرف اذا ...,10469,ما هي اعراض الغده الدرقيه !؟,4,0.8
486,9642,اعاني من نشاط في الغدة الدرقية و تسمم في الفص ...,13006,اعاني من نشاط الغدة الدرقيه,4,0.8
468,7746,اعاني من نشاط الغده الدرقيه من 15سنه وآخر طفل ...,13006,اعاني من نشاط الغدة الدرقيه,4,0.8
461,5694,اعاني من فرط نشاط الغده الدرقيه وعندي طفله ترض...,13006,اعاني من نشاط الغدة الدرقيه,4,0.8


In [122]:
category_name = "ارتفاع ضغط الدم"  #  تنفسي - دم - أطفال - أسنان - عيون - جلدية - عام - نسائية - غدد - ضغط
threshold = 0.80
min_shared_words = 4

def question_words(text):
    text = str(text).strip().lower()

    # Replace common punctuation with spaces
    for char in ".,،؛:!?؟()[]{}\"'ـ":
        text = text.replace(char, " ")

    return text.split()


def longest_common_subsequence_length(words1, words2):
    previous = [0] * (len(words2) + 1)

    for word1 in words1:
        current = [0]
        for j, word2 in enumerate(words2, start=1):
            if word1 == word2:
                current.append(previous[j - 1] + 1)
            else:
                current.append(max(previous[j], current[-1]))
        previous = current

    return previous[-1]


subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).reset_index(drop=True)
ids = subset["id"].tolist()
raw_questions = subset["question"].tolist()
words = [question_words(q) for q in subset["question"]]
word_sets = [set(w) for w in words]
lengths = [len(w) for w in words]

matches = []
n = len(ids)

for i in range(n):
    len_i = lengths[i]
    if len_i < min_shared_words:
        continue
    words_i = words[i]
    set_i = word_sets[i]

    for j in range(i + 1, n):
        len_j = lengths[j]
        if len_j < min_shared_words:
            continue

        shorter_length = len_i if len_i < len_j else len_j
        required_shared = max(min_shared_words, int(threshold * shorter_length))

        # 2. Fast upper-bound filter: unordered set intersection must meet required_shared
        if len(set_i & word_sets[j]) < required_shared:
            continue

        # 3. Only run expensive LCS on the tiny fraction of pairs that pass the filter
        shared_in_order = longest_common_subsequence_length(words_i, words[j])
        shorter_coverage = shared_in_order / shorter_length

        if shared_in_order >= min_shared_words and shorter_coverage >= threshold:
            matches.append({
                "id_1": ids[i],
                "question_1": raw_questions[i],
                "id_2": ids[j],
                "question_2": raw_questions[j],
                "shared_words_in_order": shared_in_order,
                "shorter_question_coverage": round(shorter_coverage, 3),
            })

similar_questions = pd.DataFrame(matches).sort_values(
    "shorter_question_coverage",
    ascending=False
)

display(similar_questions)

,id_1,question_1,id_2,question_2,shared_words_in_order,shorter_question_coverage
863,21785,ما هي الاطعمة التي ترفع ضغط الدم المنخفض؟ فأنا...,24625,انخفاض متكرر في ضغط الدم .,5,1.0
29,22887,ما علاج انخفاض ضغط الدم؟,23297,علاج انخفاض ضغط الدم,4,1.0
867,23872,معدل ضغط الدم الطبيعي,23593,معدل ضغط الدم الطبيعي عند الإنسان العادي وال...,4,1.0
868,23872,معدل ضغط الدم الطبيعي,21581,ما هو معدل ضغط الدم الطبيعي؟ وهل هناك أعراض أو...,4,1.0
869,23872,معدل ضغط الدم الطبيعي,25015,كم معدل ضغط الدم الطبيعي؟ في حال كان الانقباضي...,4,1.0
...,...,...,...,...,...,...
372,21956,صداع ارتفاع الضغط كيف يكون؟ فأنا أعاني من ارتف...,23213,كيف اشخص ارتفاع ضغط الدم,4,0.8
370,21827,كيف أعرف حدوث انخفاض في ضغط الدم؟ فعند تناول أ...,24625,انخفاض متكرر في ضغط الدم .,4,0.8
888,25188,ماهي اسباب ارتفاع ضغط الدم,24957,ماهي افضل حبوب لمن لديه ارتفاع ضغط الدم؟,4,0.8
887,25188,ماهي اسباب ارتفاع ضغط الدم,23644,ماهي اسباب ارتفاع ظغط الدم لدي علمي اني شاب عم...,4,0.8


<font color="orange" size=6>2- Pre-Processing Steps

<font color="#5EEDC8" size=4> إزالة الأسئلة المكررة من البيانات

In [98]:
train_df = train_df.drop_duplicates(subset=["category", "question"])

In [99]:
category_name = "أمراض الجهاز التنفسي" # تنفسي  - أطفال 

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question


In [100]:
category_name = "أمراض الأطفال" # تنفسي - دم - أطفال - أسنان

# Keep non-empty questions from the selected category
subset = train_df.loc[
    train_df["category"].eq(category_name),
    ["id", "question"]
].dropna(subset=["question"]).copy()

subset["question"] = subset["question"].astype(str).str.strip()
subset = subset[subset["question"].ne("")]

# 1. Find exact repeated questions
exact_duplicates = subset[
    subset.duplicated(subset="question", keep=False)
].sort_values("question")

print("Exact duplicates:")
display(exact_duplicates)

Exact duplicates:


,id,question


In [101]:
print("After Removing Duplicates:", train_df.shape)


After Removing Duplicates: (58113, 3)


<font color="#5EEDC8"> عمليات معالجة للنص (علامات ترقيم - تكرارات زائدة للأحرف - حذف الأرقام - توحيد بعض الأحرف المتشابهة (الألف الممدودة - الألف المقصورة))

In [102]:
# stop_words = set(stopwords.words('arabic'))

# def punctuation(t):
#     punc = '!"#$%&()*+,-./:;؛<=>؟?@[\\]^_`{|}~\t\r\n' 
#     for p in punc:                                             # 3. remove punctuation characters
#         t = t.replace(p, '')
#     return t

# def replace_alif_maksoura(t):
#     return str(t).replace("ى", "ي")

# ALIF_TRANSLATOR = str.maketrans("أإآ", "ااا")

# def normalize_alif(t):
#     return str(t).translate(ALIF_TRANSLATOR)
    
# steps = {
#     'lower':         lambda t: t.lower(),                
#     'hashtag_split': lambda t: re.sub(r'[#_]', ' ', t),            # split hashtags and underscores into separate words
#     'elongation':    lambda t: re.sub(r'(\w)\1{2,}', r'\1', t),      # letter repeated 3 or more times
#     'punctuation':   lambda t: punctuation(t),        
#     'alif_maksoura': lambda t: replace_alif_maksoura(t),
#     'alif':          lambda t: normalize_alif(t),
#     'numbers':       lambda t: re.sub(r'\d+', ' ', t),       
#     'stopwords':     lambda t: " ".join([w for w in t.split() if not w in stop_words]),    # Do we need it ?    
#     'negation_stop': lambda t: t,        # Do we need it ?
# }

<font color="#5EEDC8"> category-label mapping

In [ ]:
label_to_id = dict(zip(mapping_df["category"], mapping_df["label_id"]))
train_df["label_id"] = train_df["category"].map(label_to_id)
print(train_df.columns)
print(train_df.columns.values)
print(train_df.head(5))


Index(['id', 'question', 'category', 'label_id'], dtype='str')
<StringArray>
['id', 'question', 'category', 'label_id']
Length: 4, dtype: str
      id                                           question  \
0   5034  العمر ٤٢ سنة مريض ضغط باخد بيزوكارد10 اشعر بإر...   
1  44388  اعاني من فقر الدم . هل يمكنني ان القح نفسي بحق...   
2  37246  منذ حوالي خمسة أشهر، لاحظت نزول دم مع البراز ل...   
3   3785  انا فتاه غير متزوجة عمري 34 سنه وزني 75 سنه وط...   
4  21193  تاخرت الدورة 14 يوم وعملت اختبار دم اليوم طلع ...   

             category  label_id  
0  أمراض الغدد الصماء       4.0  
1          الطب العام       2.0  
2          الطب العام       2.0  
3          أمراض الدم       8.0  
4        أمراض نسائية       3.0  


In [108]:
mapped = train_df["category"].map(label_to_id)
if mapped.isna().any():
    missing = train_df.loc[mapped.isna(), "category"].unique()
    raise ValueError(f"Unmapped or missing categories: {missing}")
# train_df["label_id"] = train_df["label_id"].astype(int)

ValueError: Unmapped or missing categories: <StringArray>
['امراض العيون']
Length: 1, dtype: str

<font color=orange> Simple Tokenizer - Texts to Sequences - Sequences to Texts

In [104]:
# class SimpleTokenizer:
#     """A small replacement for the old Keras `Tokenizer` (it was removed in Keras 3).
#     It turns each word into a whole number. The most frequent word gets 1, the next gets 2, ...
#     The number 0 is kept for padding."""

#     def __init__(self, num_words=None):
#         self.num_words = num_words
#         self.word_index = {}      # word -> number
#         self.index_word = {}      # number -> word

#     @staticmethod
#     def _split(text):
#         text = text.lower()                                  # lower case
#         for ch in '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n':      # remove punctuation
#             text = text.replace(ch, ' ')
#         return [w for w in text.split(' ') if w]             # split into words

#     def fit_on_texts(self, texts):
#         counts = {}
#         for text in texts:
#             for w in self._split(text):
#                 counts[w] = counts.get(w, 0) + 1
#         words = sorted(counts, key=counts.get, reverse=True)  # most frequent word first
#         self.word_index = {w: i for i, w in enumerate(words, start=1)}
#         self.index_word = {i: w for w, i in self.word_index.items()}

#     def texts_to_sequences(self, texts):
#         sequences = []
#         for text in texts:
#             seq = []
#             for w in self._split(text):
#                 i = self.word_index.get(w)
#                 if i is not None and (self.num_words is None or i < self.num_words):
#                     seq.append(i)                            # unknown words are skipped
#             sequences.append(seq)
#         return sequences

#     def sequences_to_texts(self, sequences):
#         return [' '.join(self.index_word[i] for i in seq if i in self.index_word) for seq in sequences]

<font color=orange> Baseline (CountVectorizer + Logestic Regression)

In [105]:
# vectorizer = CountVectorizer()
# vectorizer.fit(X_train) # need to modify
# classifier = LogisticRegression()
# classifier.fit(vectorizer.transform(X_train), y_train) # need to modify
# score = classifier.score(vectorizer.transform(X_test), y_test) # need to modify
# print("Baseline test accuracy: {:.4f}".format(score)) # baseline score